# Exploratory Data Analysis - Code Classification Challenge

**Objectif** : Analyser en profondeur le dataset xCodeEval pour comprendre :
- La distribution des 8 tags prioritaires
- La qualité et la complétude des données
- Les patterns et caractéristiques par tag
- Les insights pour le feature engineering

**Tags prioritaires** : `math`, `graphs`, `strings`, `number theory`, `trees`, `geometry`, `games`, `probabilities`

## 1. Setup et Chargement des Données

In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuration des visualisations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Imports des fonctions utilitaires
import sys
sys.path.append('../')
from src.utils.eda_helpers import (
    load_dataset, extract_tags_list, get_tag_statistics,
    get_priority_tag_coverage, analyze_tag_cooccurrence,
    get_tag_examples, compute_text_statistics, analyze_code_complexity,
    plot_tag_distribution, plot_cooccurrence_heatmap, PRIORITY_TAGS
)

print("✅ Setup complet")
print(f"Tags prioritaires : {PRIORITY_TAGS}")

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# Chargement du dataset
DATA_DIR = '../data/raw/code_classification_dataset'
df = load_dataset(DATA_DIR)

# Affichage des premières lignes
print(f"\nShape: {df.shape}")
print(f"\nColonnes: {list(df.columns)}")
df.head()

## 2. Vue d'Ensemble du Dataset

In [ ]:
# Informations générales
print("=" * 80)
print("INFORMATIONS GÉNÉRALES")
print("=" * 80)
df.info()

In [ ]:
# Statistiques descriptives
print("\n" + "=" * 80)
print("STATISTIQUES DESCRIPTIVES")
print("=" * 80)
df.describe(include='all')

In [ ]:
# Analyse des valeurs manquantes
print("\n" + "=" * 80)
print("VALEURS MANQUANTES")
print("=" * 80)

missing = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df)) * 100
}).sort_values('Missing_Count', ascending=False)

print(missing[missing['Missing_Count'] > 0])

# Visualisation
fig, ax = plt.subplots(figsize=(10, 6))
missing_cols = missing[missing['Missing_Count'] > 0]
if len(missing_cols) > 0:
    ax.barh(missing_cols['Column'], missing_cols['Missing_Percentage'], color='#e74c3c')
    ax.set_xlabel('Pourcentage de valeurs manquantes (%)', fontsize=12)
    ax.set_title('Valeurs Manquantes par Colonne', fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("✅ Aucune valeur manquante détectée")

### Décisions de Traitement des Valeurs Manquantes

**À documenter après analyse :**
- Colonnes avec valeurs manquantes
- Impact sur l'analyse
- Stratégie de traitement (suppression, imputation, etc.)

## 3. Analyse des Tags

In [ ]:
# Parser les tags
df = extract_tags_list(df)

# Statistiques des tags
tag_stats = get_tag_statistics(df)

print("=" * 80)
print("STATISTIQUES DES TAGS")
print("=" * 80)
print(f"\nNombre total de tags uniques : {len(tag_stats)}")
print(f"\nTop 20 tags les plus fréquents :")
print(tag_stats.head(20).to_string(index=False))

In [ ]:
# Visualisation de la distribution des tags
fig = plot_tag_distribution(tag_stats, top_n=25, highlight_priority=True)
plt.show()

In [ ]:
# Focus sur les 8 tags prioritaires
priority_stats = tag_stats[tag_stats['is_priority']].copy()

print("=" * 80)
print("TAGS PRIORITAIRES - STATISTIQUES DÉTAILLÉES")
print("=" * 80)
print(priority_stats.to_string(index=False))

# Visualisation
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(priority_stats['tag'], priority_stats['count'], color='#2ecc71', alpha=0.8)
ax.set_xlabel('Tag', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Distribution des 8 Tags Prioritaires', fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Couverture des tags prioritaires
coverage = get_priority_tag_coverage(df)

print("\n" + "=" * 80)
print("COUVERTURE DES TAGS PRIORITAIRES")
print("=" * 80)
for key, value in coverage.items():
    print(f"{key}: {value}")

# Visualisation
fig, ax = plt.subplots(figsize=(8, 6))
labels = ['Avec tag prioritaire', 'Sans tag prioritaire']
sizes = [coverage['samples_with_priority_tag'], coverage['samples_without_priority_tag']]
colors = ['#2ecc71', '#e74c3c']
explode = (0.05, 0)

ax.pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%',
       shadow=True, startangle=90, textprops={'fontsize': 12})
ax.set_title('Couverture des Tags Prioritaires', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Décision Stratégique

**Question** : Faut-il filtrer le dataset pour ne garder que les exemples avec au moins un tag prioritaire ?

**À documenter** :
- Avantages/inconvénients du filtrage
- Impact sur la taille du dataset d'entraînement
- Recommandation finale

In [ ]:
# Analyse du nombre de tags par échantillon
df['num_tags'] = df['tags'].apply(len)
df['num_priority_tags'] = df['tags'].apply(lambda tags: sum(1 for t in tags if t in PRIORITY_TAGS))

print("=" * 80)
print("NOMBRE DE TAGS PAR ÉCHANTILLON")
print("=" * 80)
print(f"\nMoyenne de tags par échantillon : {df['num_tags'].mean():.2f}")
print(f"Médiane : {df['num_tags'].median():.0f}")
print(f"Min : {df['num_tags'].min()}")
print(f"Max : {df['num_tags'].max()}")

print(f"\nMoyenne de tags prioritaires par échantillon : {df['num_priority_tags'].mean():.2f}")

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['num_tags'], bins=range(1, df['num_tags'].max()+2), 
             color='#3498db', alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Nombre de tags', fontsize=12)
axes[0].set_ylabel('Fréquence', fontsize=12)
axes[0].set_title('Distribution du Nombre de Tags Total', fontsize=13, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

axes[1].hist(df['num_priority_tags'], bins=range(0, df['num_priority_tags'].max()+2), 
             color='#2ecc71', alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Nombre de tags prioritaires', fontsize=12)
axes[1].set_ylabel('Fréquence', fontsize=12)
axes[1].set_title('Distribution du Nombre de Tags Prioritaires', fontsize=13, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Co-occurrence des Tags

In [ ]:
# Matrice de co-occurrence pour les tags prioritaires
cooccurrence = analyze_tag_cooccurrence(df, PRIORITY_TAGS)

print("=" * 80)
print("MATRICE DE CO-OCCURRENCE (Tags Prioritaires)")
print("=" * 80)
print(cooccurrence)

# Visualisation
fig = plot_cooccurrence_heatmap(cooccurrence)
plt.show()

In [ ]:
# Paires de tags les plus fréquentes
from itertools import combinations
from collections import Counter

tag_pairs = []
for tags in df['tags']:
    priority_tags_in_sample = [t for t in tags if t in PRIORITY_TAGS]
    if len(priority_tags_in_sample) >= 2:
        tag_pairs.extend(list(combinations(sorted(priority_tags_in_sample), 2)))

pair_counts = Counter(tag_pairs)

print("\n" + "=" * 80)
print("TOP 10 PAIRES DE TAGS PRIORITAIRES")
print("=" * 80)
for pair, count in pair_counts.most_common(10):
    print(f"{pair[0]:20s} + {pair[1]:20s} : {count:4d} occurrences")

## 5. Analyse par Tag Prioritaire

In [ ]:
# Pour chaque tag prioritaire, analyser les caractéristiques
tag_analysis = []

for tag in PRIORITY_TAGS:
    mask = df['tags'].apply(lambda tags: tag in tags)
    tag_df = df[mask]
    
    analysis = {
        'tag': tag,
        'count': len(tag_df),
        'avg_difficulty': tag_df['difficulty'].mean() if 'difficulty' in tag_df.columns else None,
        'avg_code_length': tag_df['source_code'].str.len().mean(),
        'avg_desc_length': tag_df['prob_desc_description'].str.len().mean(),
        'most_common_lang': tag_df['lang'].mode()[0] if len(tag_df) > 0 else None
    }
    tag_analysis.append(analysis)

tag_analysis_df = pd.DataFrame(tag_analysis)

print("=" * 80)
print("ANALYSE PAR TAG PRIORITAIRE")
print("=" * 80)
print(tag_analysis_df.to_string(index=False))

In [ ]:
# Visualisation des caractéristiques par tag
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Difficulté moyenne
if tag_analysis_df['avg_difficulty'].notna().any():
    axes[0, 0].bar(tag_analysis_df['tag'], tag_analysis_df['avg_difficulty'], color='#9b59b6', alpha=0.7)
    axes[0, 0].set_ylabel('Difficulté Moyenne', fontsize=11)
    axes[0, 0].set_title('Difficulté Moyenne par Tag', fontsize=12, fontweight='bold')
    axes[0, 0].tick_params(axis='x', rotation=45)

# Longueur moyenne du code
axes[0, 1].bar(tag_analysis_df['tag'], tag_analysis_df['avg_code_length'], color='#e67e22', alpha=0.7)
axes[0, 1].set_ylabel('Longueur Moyenne (caractères)', fontsize=11)
axes[0, 1].set_title('Longueur Moyenne du Code par Tag', fontsize=12, fontweight='bold')
axes[0, 1].tick_params(axis='x', rotation=45)

# Longueur moyenne de la description
axes[1, 0].bar(tag_analysis_df['tag'], tag_analysis_df['avg_desc_length'], color='#1abc9c', alpha=0.7)
axes[1, 0].set_ylabel('Longueur Moyenne (caractères)', fontsize=11)
axes[1, 0].set_title('Longueur Moyenne de la Description par Tag', fontsize=12, fontweight='bold')
axes[1, 0].tick_params(axis='x', rotation=45)

# Nombre d'échantillons
axes[1, 1].bar(tag_analysis_df['tag'], tag_analysis_df['count'], color='#34495e', alpha=0.7)
axes[1, 1].set_ylabel('Nombre d\'échantillons', fontsize=11)
axes[1, 1].set_title('Nombre d\'Échantillons par Tag', fontsize=12, fontweight='bold')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 6. Analyse NLP des Descriptions

In [ ]:
# Installation de wordcloud si nécessaire
try:
    from wordcloud import WordCloud
except ImportError:
    print("Installation de wordcloud...")
    !pip install wordcloud
    from wordcloud import WordCloud

import re
from collections import Counter

In [ ]:
def clean_text(text):
    """Nettoyer le texte pour l'analyse NLP"""
    # Supprimer les formules LaTeX
    text = re.sub(r'\$\$\$.*?\$\$\$', '', text)
    # Supprimer les caractères spéciaux
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    # Convertir en minuscules
    text = text.lower()
    # Supprimer les espaces multiples
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Nettoyer les descriptions
df['clean_description'] = df['prob_desc_description'].apply(clean_text)

print("✅ Descriptions nettoyées")

In [ ]:
# Nuages de mots par tag prioritaire
fig, axes = plt.subplots(4, 2, figsize=(16, 20))
axes = axes.flatten()

for idx, tag in enumerate(PRIORITY_TAGS):
    # Récupérer les descriptions pour ce tag
    mask = df['tags'].apply(lambda tags: tag in tags)
    tag_descriptions = ' '.join(df[mask]['clean_description'].tolist())
    
    if len(tag_descriptions.strip()) > 0:
        # Générer le nuage de mots
        wordcloud = WordCloud(width=800, height=400, 
                              background_color='white',
                              colormap='viridis',
                              max_words=100).generate(tag_descriptions)
        
        axes[idx].imshow(wordcloud, interpolation='bilinear')
        axes[idx].set_title(f'Tag: {tag}', fontsize=14, fontweight='bold')
        axes[idx].axis('off')
    else:
        axes[idx].text(0.5, 0.5, f'Pas de données pour {tag}', 
                      ha='center', va='center', fontsize=12)
        axes[idx].axis('off')

plt.tight_layout()
plt.savefig('../docs/wordclouds_by_tag.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Nuages de mots générés et sauvegardés")

In [ ]:
# Mots-clés les plus fréquents par tag
from sklearn.feature_extraction.text import CountVectorizer

print("=" * 80)
print("TOP 15 MOTS-CLÉS PAR TAG PRIORITAIRE")
print("=" * 80)

for tag in PRIORITY_TAGS:
    mask = df['tags'].apply(lambda tags: tag in tags)
    tag_descriptions = df[mask]['clean_description'].tolist()
    
    if len(tag_descriptions) > 0:
        # Vectorisation
        vectorizer = CountVectorizer(max_features=15, stop_words='english')
        try:
            X = vectorizer.fit_transform(tag_descriptions)
            word_counts = X.sum(axis=0).A1
            words = vectorizer.get_feature_names_out()
            
            print(f"\n{tag.upper()}:")
            for word, count in sorted(zip(words, word_counts), key=lambda x: x[1], reverse=True):
                print(f"  {word:20s}: {count:5d}")
        except:
            print(f"\n{tag.upper()}: Erreur lors de la vectorisation")

## 7. Analyse du Code Source

In [ ]:
# Statistiques de longueur du code
code_stats = compute_text_statistics(df['source_code'])
df = pd.concat([df, code_stats], axis=1)

print("=" * 80)
print("STATISTIQUES DU CODE SOURCE")
print("=" * 80)
print(code_stats.describe())

In [ ]:
# Visualisation de la distribution de la longueur du code
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].hist(df['length_chars'], bins=50, color='#3498db', alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Nombre de caractères', fontsize=11)
axes[0].set_ylabel('Fréquence', fontsize=11)
axes[0].set_title('Distribution - Longueur en Caractères', fontsize=12, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

axes[1].hist(df['length_words'], bins=50, color='#e74c3c', alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Nombre de mots', fontsize=11)
axes[1].set_ylabel('Fréquence', fontsize=11)
axes[1].set_title('Distribution - Longueur en Mots', fontsize=12, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

axes[2].hist(df['length_lines'], bins=50, color='#2ecc71', alpha=0.7, edgecolor='black')
axes[2].set_xlabel('Nombre de lignes', fontsize=11)
axes[2].set_ylabel('Fréquence', fontsize=11)
axes[2].set_title('Distribution - Longueur en Lignes', fontsize=12, fontweight='bold')
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Analyse de complexité du code (échantillon)
print("Analyse de la complexité du code (sur un échantillon de 100 exemples)...")

sample_df = df.sample(min(100, len(df)), random_state=42)
complexity_results = sample_df['source_code'].apply(analyze_code_complexity)
complexity_df = pd.DataFrame(complexity_results.tolist())

print("\n" + "=" * 80)
print("STATISTIQUES DE COMPLEXITÉ DU CODE (Échantillon)")
print("=" * 80)
print(complexity_df.describe())

## 8. Distribution des Langages

In [ ]:
# Distribution des langages
lang_counts = df['lang'].value_counts()

print("=" * 80)
print("DISTRIBUTION DES LANGAGES")
print("=" * 80)
print(lang_counts)

# Visualisation
fig, ax = plt.subplots(figsize=(10, 6))
lang_counts.plot(kind='bar', ax=ax, color='#9b59b6', alpha=0.7, edgecolor='black')
ax.set_xlabel('Langage', fontsize=12)
ax.set_ylabel('Nombre d\'échantillons', fontsize=12)
ax.set_title('Distribution des Langages de Programmation', fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Exemples Représentatifs par Tag

In [ ]:
# Afficher des exemples pour chaque tag prioritaire
for tag in PRIORITY_TAGS:
    print("\n" + "=" * 80)
    print(f"EXEMPLES POUR LE TAG: {tag.upper()}")
    print("=" * 80)
    
    examples = get_tag_examples(df, tag, n=2)
    
    for idx, row in examples.iterrows():
        print(f"\nExemple {idx + 1}:")
        print(f"Tags: {row['tags']}")
        print(f"Difficulté: {row['difficulty']}")
        print(f"\nDescription (extrait):")
        print(row['prob_desc_description'][:300] + "...")
        print(f"\nCode (extrait):")
        print(row['source_code'][:200] + "...")
        print("-" * 80)

## 10. Synthèse et Recommandations

### Résumé des Findings

**À compléter après analyse :**

1. **Qualité des données**
   - Valeurs manquantes identifiées
   - Décisions de preprocessing

2. **Distribution des tags**
   - Tags les plus/moins fréquents
   - Déséquilibre des classes
   - Couverture des tags prioritaires

3. **Patterns identifiés**
   - Co-occurrences significatives
   - Mots-clés discriminants par tag
   - Caractéristiques du code par tag

4. **Recommandations pour le Feature Engineering**
   - Features textuelles à extraire
   - Features de code à calculer
   - Stratégies de preprocessing

5. **Recommandations pour la Modélisation**
   - Gestion du déséquilibre
   - Approche multi-label
   - Métriques d'évaluation

In [ ]:
# Sauvegarder le dataset enrichi pour les prochaines étapes
output_path = '../data/processed/dataset_with_eda_features.parquet'
df.to_parquet(output_path, index=False)
print(f"✅ Dataset enrichi sauvegardé: {output_path}")